In [2]:
import os
os.chdir(r"C:\Users\lalit\Documents\nyc-congestion-toll-impact")
print(os.getcwd())

C:\Users\lalit\Documents\nyc-congestion-toll-impact


In [4]:
import os
import duckdb
import pandas as pd

os.chdir(r"C:\Users\lalit\Documents\nyc-congestion-toll-impact")
pd.set_option("display.width", 200)

yoy = duckdb.sql("""
    WITH base AS (
        SELECT CASE WHEN trip_date < DATE '2025-01-01' THEN 'before' ELSE 'after' END AS period,
               company, trip_group,
               SUM(trips) AS trips,
               SUM(sum_rider_cost) / SUM(trips) AS avg_cost,
               SUM(sum_miles) / (SUM(sum_trip_seconds) / 3600) AS mph,
               SUM(sum_driver_pay) / (SUM(sum_trip_seconds) / 3600) AS driver_pay_hr,
               SUM(sum_wait_seconds) / SUM(wait_trips) / 60 AS wait_min
        FROM 'data/summary/*.parquet'
        WHERE (trip_date BETWEEN '2024-02-01' AND '2024-07-31')
           OR (trip_date BETWEEN '2025-02-01' AND '2025-07-31')
        GROUP BY ALL
    )
    SELECT a.company, a.trip_group,
           ROUND(100 * (a.trips / b.trips - 1), 1)                 AS trips_pct,
           ROUND(100 * (a.avg_cost / b.avg_cost - 1), 1)           AS cost_pct,
           ROUND(100 * (a.mph / b.mph - 1), 1)                     AS speed_pct,
           ROUND(100 * (a.driver_pay_hr / b.driver_pay_hr - 1), 1) AS pay_hr_pct,
           ROUND(a.wait_min - b.wait_min, 2)                       AS wait_change_min
    FROM base a JOIN base b USING (company, trip_group)
    WHERE a.period = 'after' AND b.period = 'before'
      AND a.company IN ('Uber', 'Lyft', 'Yellow')
    ORDER BY a.company, a.trip_group
""").df()

yoy

,company,trip_group,trips_pct,cost_pct,speed_pct,pay_hr_pct,wait_change_min
0,Lyft,boundary,12.4,1.7,-0.9,3.7,0.07
1,Lyft,entering,12.5,0.8,-0.0,4.2,0.08
2,Lyft,exiting,3.6,1.3,0.4,4.2,0.13
3,Lyft,inside,10.3,7.0,3.6,4.5,0.02
4,Lyft,outside,10.0,-3.4,-2.6,2.4,0.13
5,Uber,boundary,-10.9,11.8,2.8,7.0,0.06
6,Uber,entering,-11.1,13.1,3.4,8.0,0.39
7,Uber,exiting,-7.2,12.4,3.2,8.2,0.14
8,Uber,inside,-13.4,18.5,1.0,8.3,-0.10
9,Uber,outside,0.2,2.2,-0.8,3.0,0.37
